In [0]:
%sql
SHOW TABLES IN formula1_catalog.bronze

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.sprints"
silver_table=f"{catalog_name}.{silver_schema}.sprints"


In [0]:

bronze_table

In [0]:
%sql
describe history formula1_catalog.bronze.sprints
--describe history formula1_catalog.SILVER.sprints

In [0]:
# spark.read for aditonal options to read table data
#ciucuits_df=spark.read.option('versionAsOf',0).table(bronze_table)

In [0]:
df=spark.read.table(bronze_table)

In [0]:
sprints_df = (
    spark.table(bronze_table)
    .select(
        "season",
        "round",
        "constructorId",
        "driverId",
        "date",
        "raceName",
        "grid",
        "laps",
        "number",
        "points",
        "position",
        "positionText",
        "status",
        "ingestion_timestamp",
        "source_file",
    )
    .withColumnsRenamed(
        {
            "driverId": "driver_id",
            "date": "race_date",
            "constructorId": "constructor_id",
            "raceName": "race_name",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "points": "race_points",
            "position": "final_position",
            "positionText": "final_position_text",
        }
    )
)

In [0]:
from pyspark.sql import functions as F

sprints_valid_df=(sprints_df.
    filter(
    F.col('season').isNotNull()&
    F.col('round').isNotNull()&
    F.col('constructor_id').isNotNull() &
    F.col('driver_id').isNotNull() 
    )
    .dropDuplicates(["driver_id", "season", "round", "constructor_id"])
)

In [0]:
display(sprints_df.count()- sprints_valid_df.count())

In [0]:
from pyspark.sql.functions import initcap
sprints_final_df=(sprints_valid_df
    .withColumn('race_name',F.initcap(F.col('race_name')))
    
 )

In [0]:
display(sprints_final_df)

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode('overwrite')
        .option("mergeSchema", "true")
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1_catalog.silver.sprints